## Quality of Care Indicators

Compute district-year quality-of-care indicators from DHIS2 routine data produced by outliers pipelines (`imputed` or `removed`).

Indicators:
- testing_rate = TEST / SUSP
- treatment_rate = MALTREAT / CONF
- case_fatality_rate = MALDTH / MALADM
- prop_adm_malaria = MALADM / ALLADM
- prop_malaria_deaths = MALDTH / ALLDTH
- non_malaria_all_cause_outpatients = ALLOUT (absolute)
- presumed_cases = PRES (absolute)

Stock-out indicators are not implemented yet (on hold, NMDR data pending).

In [ ]:
# Preliminaries
options(scipen = 999)

ROOT_PATH <- "~/workspace"
source(file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care", "utils", "snt_dhis2_quality_of_care.r"))

snt_environment <- get_setup_variables(SNT_ROOT_PATH = ROOT_PATH, packages = c("jsonlite", "data.table", "arrow", "sf", "ggplot2", "glue", "reticulate", "RColorBrewer", "dplyr", "writexl", "knitr", "scales", "gridExtra"))
config_json <- load_snt_config(snt_environment$CONFIG_PATH, "SNT_config.json")
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
DHIS2_FORMATTED_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
OUTLIERS_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_OUTLIERS_IMPUTATION

PIPELINE_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care")
OUTPUT_DATA_PATH <- file.path(ROOT_PATH, "data", "dhis2", "quality_of_care")
REPORT_OUTPUTS_PATH <- file.path(PIPELINE_PATH, "reporting", "outputs")
FIGURES_PATH <- file.path(REPORT_OUTPUTS_PATH, "figures")
dir.create(OUTPUT_DATA_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(REPORT_OUTPUTS_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_PATH, recursive = TRUE, showWarnings = FALSE)

In [ ]:
# Load and prepare inputs
if (!exists("data_action")) {
  data_action <- "imputed"
}
data_action <- validate_quality_of_care_action(data_action)

log_msg(glue::glue("Using outliers dataset id: {OUTLIERS_DATASET}"))

routine_filename <- glue::glue("{COUNTRY_CODE}_routine_outliers_{data_action}.parquet")
log_msg(glue::glue("Using routine file: {routine_filename}"))

routine <- load_country_file_from_dataset(
  dataset_id = OUTLIERS_DATASET,
  country_code = COUNTRY_CODE,
  suffix = glue::glue("_routine_outliers_{data_action}.parquet"),
  label = "Quality of care routine"
)

shapes <- load_country_file_from_dataset(
  dataset_id = DHIS2_FORMATTED_DATASET,
  country_code = COUNTRY_CODE,
  suffix = "_shapes.geojson",
  label = "DHIS2 shapes data"
)

# Compute indicators
core_cols <- c("ADM2_ID", "YEAR")
core_missing <- setdiff(core_cols, names(routine))
if (length(core_missing) > 0) {
  stop(glue::glue("[ERROR] Missing core columns: {paste(core_missing, collapse = ', ')}"))
}

indicator_cols <- c("TEST", "SUSP", "MALTREAT", "CONF", "MALDTH", "MALADM", "ALLADM", "ALLDTH", "ALLOUT", "PRES")
missing_cols <- setdiff(indicator_cols, names(routine))
if (length(missing_cols) > 0) {
  log_msg(glue::glue("[WARNING] Missing indicator columns: {paste(missing_cols, collapse = ', ')}"), level = "warning")
}

routine <- normalize_qoc_routine_types(routine)
qoc <- aggregate_qoc_district_year(routine)
qoc <- add_quality_of_care_derived_indicators(qoc)

# Enrich labels
qoc <- attach_quality_of_care_shapes(qoc, shapes)

# Save district-year outputs
save_quality_of_care_outputs(
  qoc_dt = qoc,
  output_data_path = OUTPUT_DATA_PATH,
  country_code = COUNTRY_CODE,
  data_action = data_action
)

In [ ]:
# Build yearly maps (saved as PNG)
save_quality_of_care_maps(
  qoc_dt = qoc,
  shapes_sf = shapes,
  figures_path = FIGURES_PATH
)